<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/GauStudio_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## GauStudio — 3DGS to Mesh via TSDF Fusion (MIT + INRIA mixed)

**License notice:** [GauStudio](https://github.com/GAP-LAB-CUHK-SZ/gaustudio) (Chongjie Ye et al., GAP Lab @ CUHK-Shenzhen, [arXiv 2403.19632](https://arxiv.org/abs/2403.19632)) is **MIT-licensed for the main framework**, but the bundled `gaustudio-diff-gaussian-rasterization` is a derivative of INRIA's rasterizer and inherits its **non-commercial** restriction. This is the same licensing situation as [SuGaR](https://github.com/Anttwo/SuGaR). Like SuGaR, included in AEI-Colab-Notebooks for **personal-asset research and evaluation only**.

**What it does:** Takes a trained 3D Gaussian Splatting scene (or a TripoSplat PLY) and extracts a clean mesh by **rendering depth maps from the 3DGS cameras + fusing them with TSDF** (truncated signed distance function) via [vdbfusion](https://github.com/nicolamattina/vdbfusion). This is a fundamentally different algorithm family from TripoSplat's Poisson reconstruction or SuGaR's density-field approach.

**Why use this instead of TripoSplat's default or SuGaR?**

- **vs TripoSplat's Poisson mesh**: GauStudio uses rendered depth maps (not just point samples) so the mesh follows the actual 3DGS surface more faithfully. Cleaner topology, fewer holes.
- **vs SuGaR**: GauStudio is **much faster** (~5-10 min vs 2-3 hrs per scene) and runs on **T4 15 GB** (SuGaR needs L4 22 GB). Trade-off: GauStudio's mesh is **smoother** but less geometrically faithful than SuGaR. For low-LOD game assets the smoother output is actually preferred.
- **vs commercial (Kiri Engine, Polycam)**: GauStudio is free, runs in Colab, and you keep the data. Kiri's commercial "3DGS-to-Mesh 2.0" is built in partnership with GauStudio's author; same algorithmic family.

**What this notebook does:**

- **Step 1** — install pinned PyTorch 2.0.1+cu118 + clone GauStudio + build `vdbfusion` from source + build `gaustudio-diff-gaussian-rasterization` CUDA extension + `pip install -e .`
- **Step 2** — Drive cache prologue + verify install + GPU check
- **Step 3** — **TripoSplat-PLY-to-GauStudio-scene bridge**: takes a TripoSplat-generated 3DGS PLY + preprocessed image, builds the directory structure GauStudio expects (`cameras.json` + `point_cloud/iteration_7000/point_cloud.ply`). Same bridge concept as SuGaR's Step 2 but produces GauStudio's custom `cameras.json` format.
- **Step 4** — **Optional multi-view scene support**: if you have 3+ overlapping views of a subject (from photogrammetry, video, or multi-shot TripoSplat), you can skip the bridge and provide a real 3DGS scene with COLMAP cameras. This unlocks GauStudio's full quality.
- **Step 5** — run `gs-extract-mesh` (CLI, single command)
- **Step 6** — verify + game-asset transform (scale to meters, base-pivot center) + export `.glb` for game engines
- **Step 7** — keep-alive (the 5-10 min extraction is short, but good to have)
- **Step 8** — Drive mirror + download links
- **Step 9** — help / license / known issues / citation

**Compute:** T4 15 GB works (GauStudio's minimum is 6 GB). L4 / A100 give headroom. First run: ~10 min install + ~5-10 min extract. Subsequent runs: ~3 min install + ~5 min extract.

**Limitations:** The texturing step uses `mvs-texturing` (C++ build, brittle on Colab) and is **skipped** in this notebook. You get an untextured mesh; texture it in Blender or a game engine. This is on purpose — a textured mesh from one input image is always going to be smeared; better to texture properly in the target engine.

In [ ]:
# @title STEP 1 — Install Dependencies (GauStudio + vdbfusion + custom rasterizer)
# @markdown Installs the GauStudio framework plus its CUDA extension. ~5-10 min
# @markdown on first run, ~2 min on subsequent runs. Key dependencies:
# @markdown PyTorch 2.0.1+cu118 (pinned; the custom rasterizer must compile
# @markdown against the active PyTorch/CUDA), vdbfusion (TSDF fusion, built
# @markdown from source because PyPI wheels are missing for some Python versions),
# @markdown and the gaustudio-diff-gaussian-rasterization CUDA extension (must be
# @markdown built locally against the active PyTorch).

import os, sys, subprocess, time

print('[gaus] GauStudio + INRIA-rasterizer license notice')
print('[gaus] Main framework: MIT. Rasterizer: INRIA non-commercial.')
print('[gaus] Free for personal-asset research/evaluation. NOT for commercial use.')
print('[gaus] See: https://github.com/GAP-LAB-CUHK-SZ/gaustudio/blob/main/LICENSE')
print('[gaus] If you ship a commercial product, do NOT use this notebook.')
print(flush=True)

REPO_DIR = '/content/gaustudio'
REPO_URL = 'https://github.com/GAP-LAB-CUHK-SZ/gaustudio.git'
if not os.path.isdir(REPO_DIR):
    print(f'[git] Cloning {REPO_URL} (depth=1)...', flush=True)
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    print(f'[git] Cloned to {REPO_DIR}', flush=True)
else:
    print(f'[git] {REPO_DIR} already present, skipping clone.', flush=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# ── Pinned PyTorch 2.0.1 + CUDA 11.8 (MUST come first) ──────────────────────────
# Colab 2026.01 ships torch 2.9 / cu126 by default. The custom rasterizer
# MUST be built against the active PyTorch, so we swap to a pinned version
# first to ensure the build is reproducible. After install, the rasterizer
# build (below) will compile against this exact torch+CUDA combo.
t0 = time.time()
print('[pip] Installing pinned PyTorch 2.0.1+cu118 ...', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                'torch==2.0.1+cu118', 'torchvision==0.15.2+cu118', 'torchaudio==2.0.2+cu118',
                '--index-url', 'https://download.pytorch.org/whl/cu118'], check=True)
print(f'[pip] torch swap: {time.time()-t0:.1f}s', flush=True)

# ── GauStudio Python deps (from requirements.txt) ──────────────────────────────────
t0 = time.time()
print('[pip] Installing GauStudio Python deps ...', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'plyfile', 'tqdm', 'opencv-python-headless', 'trimesh',
                'omegaconf', 'einops', 'kiui', 'scipy', 'click',
                'open3d', 'pyexr', 'rembg', 'gradio==5.49.1'],
               check=True)
print(f'[pip] Python deps: {time.time()-t0:.1f}s', flush=True)

# ── vdbfusion (built from source — PyPI wheel missing on some Pythons) ─────────
# vdbfusion is the TSDF (truncated signed distance function) fuser that
# GauStudio uses to combine the rendered depth maps into a single mesh.
# It's a C++/pybind11 build. Compiles in 2-4 min on Colab.
VDBFUSION_DIR = '/content/vdbfusion'
VDBFUSION_URL = 'https://github.com/nicolamattina/vdbfusion.git'
if not os.path.isdir(VDBFUSION_DIR):
    print('[git] Cloning vdbfusion ...', flush=True)
    subprocess.run(['git', 'clone', '--depth', '1', VDBFUSION_URL, VDBFUSION_DIR], check=True)
t0 = time.time()
print('[build] Compiling vdbfusion C++ extension (2-4 min) ...', flush=True)
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', VDBFUSION_DIR], check=False)
if result.returncode == 0:
    print(f'[build] vdbfusion: OK ({time.time()-t0:.1f}s)', flush=True)
else:
    print(f'[build] vdbfusion: FAILED (returncode={result.returncode})', flush=True)
    print('[build] Falling back to PyPI vdbfusion (may not exist for your Python)...', flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vdbfusion'], check=False)

# ── Build custom rasterizer (MUST compile against active PyTorch) ──────────────
# The gaustudio-diff-gaussian-rasterization is a CUDA extension, a derivative
# of the INRIA 3DGS rasterizer. It MUST be built against the installed
# PyTorch. This is the #1 source of failures — the .so will not load if
# torch ABI changes. 2-5 min build time.
rasterizer_dir = os.path.join(REPO_DIR, 'submodules', 'gaustudio-diff-gaussian-rasterization')
if os.path.isdir(rasterizer_dir):
    t0 = time.time()
    print(f'[build] Compiling {rasterizer_dir} (2-5 min) ...', flush=True)
    result = subprocess.run([sys.executable, 'setup.py', 'install'], cwd=rasterizer_dir, check=False)
    if result.returncode == 0:
        print(f'[build] rasterizer: OK ({time.time()-t0:.1f}s)', flush=True)
    else:
        print(f'[build] rasterizer: FAILED (returncode={result.returncode})', flush=True)
        print('[build] Common cause: torch version mismatch. Make sure PyTorch==2.0.1+cu118 is active.', flush=True)
else:
    print(f'[build] WARNING: {rasterizer_dir} not found', flush=True)

# ── Install GauStudio as editable package (registers CLI commands) ────────────
t0 = time.time()
print('[pip] Installing gauStudio as editable package ...', flush=True)
result = subprocess.run([sys.executable, 'setup.py', 'develop'], cwd=REPO_DIR, check=False)
if result.returncode == 0:
    print(f'[pip] gauStudio develop install: OK ({time.time()-t0:.1f}s)', flush=True)
else:
    print(f'[pip] gauStudio develop install: FAILED (returncode={result.returncode})', flush=True)
    print('[pip] Falling back to: pip install -e . ...', flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO_DIR, check=False)

# ── Sanity check ─────────────────────────────────────────────────────
import torch
print(f'[verify] torch={torch.__version__} cuda={torch.cuda.is_available()}', flush=True)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'[verify] GPU: {gpu} ({vram:.1f} GB)', flush=True)
    if vram < 5:
        print(f'[verify] WARNING: only {vram:.1f} GB VRAM. GauStudio needs ~6 GB minimum.', flush=True)
    else:
        print(f'[verify] VRAM OK for GauStudio mesh extraction.', flush=True)
try:
    from diff_gaussian_rasterization import GaussianRasterizationSettings, GaussianRasterizer
    print('[verify] diff_gaussian_rasterization OK', flush=True)
except ImportError as e:
    print(f'[verify] diff_gaussian_rasterization FAILED: {e}', flush=True)
    print('[verify] Re-run the rasterizer build in this cell.', flush=True)
try:
    import vdbfusion
    print(f'[verify] vdbfusion OK', flush=True)
except ImportError as e:
    print(f'[verify] vdbfusion FAILED: {e}', flush=True)
try:
    import open3d, trimesh, plyfile
    print(f'[verify] open3d={open3d.__version__} trimesh={trimesh.__version__} OK', flush=True)
except ImportError as e:
    print(f'[verify] mesh deps FAILED: {e}', flush=True)

# ── Verify CLI entry points ────────────────────────────────────────────────────
import shutil
clis = ['gs-extract-mesh', 'gs-process-data', 'gs-render', 'gs-texture-mesh', 'gs-extract-pcd', 'gs-from-mesh', 'gs-init']
for cli in clis:
    found = shutil.which(cli) is not None
    print(f'[verify] CLI {cli}: {"OK" if found else "NOT FOUND"}', flush=True)

print(f'\n[Done] STEP 1 complete. Move on to STEP 2 (Drive cache + GPU check).', flush=True)


In [ ]:
# @title STEP 2 — Drive Cache + GPU Check
# @markdown Sets up the Drive cache for GauStudio outputs and verifies the
# @markdown GPU is ready. Outputs go to
# @markdown `/content/drive/MyDrive/AEI_3D_Out/GauStudio/`.

import os, sys, time, pathlib, subprocess
os.chdir('/content/gaustudio')

DRIVE_BASE = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/GauStudio')
DRIVE_BASE.parent.mkdir(parents=True, exist_ok=True)

if not DRIVE_BASE.parent.exists():
    print('[cache] Drive not mounted. Outputs will only be in /content/gau_out/')
    print('[cache]   Mount Drive for persistence: from the Colab file browser, click the Drive icon.')
else:
    print(f'[cache] Drive cache: {DRIVE_BASE.parent}')
    print(f'[cache] Outputs will be mirrored to {DRIVE_BASE} at the end.')

# Verify GPU + custom rasterizer still loaded (Step 1's installs persist
# across cell runs in the same session, but can be lost on reconnect).
import torch
print(f'\n[verify] torch={torch.__version__} cuda={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'[verify] GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB)')
else:
    raise SystemExit('[verify] No GPU detected. GauStudio needs CUDA. Reconnect with GPU runtime.')

try:
    from diff_gaussian_rasterization import GaussianRasterizationSettings, GaussianRasterizer
    print('[verify] diff_gaussian_rasterization OK')
except ImportError as e:
    print(f'[verify] diff_gaussian_rasterization FAILED: {e}')
    print('[verify] Re-run STEP 1.')
    raise

try:
    import vdbfusion
    print('[verify] vdbfusion OK')
except ImportError as e:
    print(f'[verify] vdbfusion FAILED: {e}')
    print('[verify] Re-run STEP 1.')
    raise

print('\n[Done] STEP 2 complete. Move on to STEP 3 (TripoSplat bridge) or STEP 4 (multi-view scene).')


In [ ]:
# @title STEP 3 — TripoSplat-PLY-to-GauStudio-Scene Bridge (single-image input)
# @markdown GauStudio expects a directory with a `cameras.json` and a
# @markdown `point_cloud/iteration_7000/point_cloud.ply`. For a single image
# @markdown (your 200+ library use case) we build this from a TripoSplat
# @markdown output. The camera is a single virtual pinhole estimated from
# @markdown TripoSplat's preprocessing. **Quality gain over TripoSplat's
# @markdown default mesh will be modest for single-view; dramatic for multi-view
# @markdown (use STEP 4 instead).**

import os, sys, time, json, shutil, pathlib
import numpy as np
import torch
from PIL import Image
from scipy.spatial.transform import Rotation

# ── Input ──────────────────────────────────────────────────────────────
TRIPOSPLAT_PLY = '/content/triposplat_runs/quicktest/quick.ply'  # @param {type:"string"}
# @markdown *Path to a TripoSplat-generated .ply (the 3DGS Gaussian PLY, NOT the _mesh.ply).*
TRIPOSPLAT_IMAGE = '/content/triposplat_runs/quicktest/preprocessed.png'  # @param {type:"string"}
# @markdown *Path to the preprocessed 1024×1024 image. Used for camera intrinsics.*
SCENE_NAME = 'triposplat_bridge'  # @param {type:"string"}
# @markdown *Output scene name. Will be created under /content/gau_out/<name>/.*
OUTPUT_DIR = f'/content/gau_out/{SCENE_NAME}'
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

ply_path = pathlib.Path(TRIPOSPLAT_PLY)
img_path = pathlib.Path(TRIPOSPLAT_IMAGE)
if not ply_path.exists():
    raise SystemExit(f'[bridge] TripoSplat PLY not found: {ply_path}')
if not img_path.exists():
    raise SystemExit(f'[bridge] Preprocessed image not found: {img_path}')
print(f'[bridge] TripoSplat PLY: {ply_path} ({ply_path.stat().st_size/1024/1024:.1f} MB)')
print(f'[bridge] Preprocessed image: {img_path}')

# ── Build GauStudio's expected directory structure ───────────────────────────
ITERATION = 7000  # GauStudio looks for iteration_7000/ by default
pc_dir = pathlib.Path(OUTPUT_DIR) / 'point_cloud' / f'iteration_{ITERATION}'
pc_dir.mkdir(parents=True, exist_ok=True)
dest_ply = pc_dir / 'point_cloud.ply'
shutil.copy2(ply_path, dest_ply)
print(f'[bridge] Copied 3DGS PLY to {dest_ply}')

# ── Build cameras.json (GauStudio's custom format) ───────────────────────────────
# GauStudio's cameras.json is a list of per-image entries with R, T, fx, fy,
# cx, cy, image_name, image_path, width, height. We build a single-entry
# list for the one image. Camera intrinsics are estimated from TripoSplat's
# preprocessing (1024x1024, ~50° FoV). Extrinsics: place camera at z=2.5
# looking at the origin (matching TripoSplat's [-0.5, 0.5]³ aabb).
with Image.open(img_path) as im:
    W, H = im.size
fov_y_deg = 50.0
fy = (H / 2.0) / np.tan(np.deg2rad(fov_y_deg / 2.0))
fx = fy
cx, cy = W / 2.0, H / 2.0

# Camera-to-world transform (3DGS convention: +X right, +Y up, +Z forward)
cam_pos = np.array([0.0, 0.0, 2.5])
forward = -cam_pos / np.linalg.norm(cam_pos)
world_up = np.array([0.0, 1.0, 0.0])
right = np.cross(forward, world_up); right /= np.linalg.norm(right)
up = np.cross(right, forward)
R_c2w = np.stack([right, up, -forward], axis=1)  # 3x3 world-from-camera rotation
t_c2w = cam_pos.reshape(3, 1)
# GauStudio expects R and T as flat lists
R_list = R_c2w.flatten().tolist()
T_list = t_c2w.flatten().tolist()

camera_entry = {
    'R': R_list,           # 9 floats, world-from-camera rotation, row-major
    'T': T_list,           # 3 floats, world-from-camera translation
    'fx': fx, 'fy': fy, 'cx': cx, 'cy': cy,
    'image_name': img_path.name,
    'image_path': str(img_path),
    'width': W, 'height': H,
}
cameras_json = [camera_entry]  # list, one entry per image
with open(pathlib.Path(OUTPUT_DIR) / 'cameras.json', 'w') as f:
    json.dump(cameras_json, f, indent=2)
print(f'[bridge] Wrote cameras.json with 1 entry: {img_path.name}')
print(f'[bridge] Camera: fx={fx:.1f} fy={fy:.1f} cx={cx:.1f} cy={cy:.1f} (FoV_y={fov_y_deg}°)')
print(f'[bridge] Camera pose: pos={tuple(cam_pos.round(3))}, looking at origin')

# ── Also copy the image to the scene (GauStudio needs it for rendering) ──
img_dir = pathlib.Path(OUTPUT_DIR) / 'images'
img_dir.mkdir(exist_ok=True)
shutil.copy2(img_path, img_dir / img_path.name)
print(f'[bridge] Copied image to {img_dir / img_path.name}')

print(f'\n[Done] Bridge complete. Scene at: {OUTPUT_DIR}')
print(f'[Done] Move on to STEP 5 (gs-extract-mesh).')


In [ ]:
# @title STEP 4 — Multi-View Scene Support (skip if using STEP 3)
# @markdown If you have a **real 3DGS scene** (not just a TripoSplat PLY) with
# @markdown **3+ overlapping views of the same subject**, you can skip STEP 3
# @markdown and use this cell to point at your existing scene. This unlocks
# @markdown GauStudio's full quality. The directory must already contain:
# @markdown   - `cameras.json` (GauStudio format, not COLMAP cameras.txt)
# @markdown   - `point_cloud/iteration_7000/point_cloud.ply`
# @markdown   - `images/<name>.png`
#
# @markdown **Skip this cell entirely** if you used STEP 3 (TripoSplat bridge).
# @markdown The variable SCENE_DIR set here overrides STEP 3's output.

import os, json, pathlib, shutil

USE_MULTIVIEW = False  # @param {type:"boolean"}
# @markdown *Set to True if you have a real multi-view 3DGS scene. False to use STEP 3's bridge output.*
MULTIVIEW_SCENE_DIR = '/content/my_3dgs_scene'  # @param {type:"string"}
# @markdown *Path to your existing 3DGS scene. Must have cameras.json + point_cloud/iteration_7000/point_cloud.ply + images/.*

if not USE_MULTIVIEW:
    print('[multiview] Skipping (USE_MULTIVIEW=False). Using STEP 3 bridge output.')
    print('[multiview] Move on to STEP 5.')
    SCENE_DIR = OUTPUT_DIR  # from STEP 3
else:
    SCENE_DIR = MULTIVIEW_SCENE_DIR
    if not os.path.isdir(SCENE_DIR):
        raise SystemExit(f'[multiview] {SCENE_DIR} not found')
    # Validate
    for sub in ('cameras.json',
                f'point_cloud/iteration_7000/point_cloud.ply'):
        if not os.path.exists(os.path.join(SCENE_DIR, sub)):
            raise SystemExit(f'[multiview] Missing {sub} in {SCENE_DIR}')
    if not os.path.isdir(os.path.join(SCENE_DIR, 'images')):
        raise SystemExit(f'[multiview] Missing images/ in {SCENE_DIR}')
    with open(os.path.join(SCENE_DIR, 'cameras.json')) as f:
        cams = json.load(f)
    print(f'[multiview] Loaded {len(cams)} cameras from {SCENE_DIR}')
    print(f'[multiview] Image: {cams[0]["image_name"]} ({cams[0]["width"]}x{cams[0]["height"]})')
    print('[multiview] Move on to STEP 5.')

# Make SCENE_DIR available globally for STEP 5
OUTPUT_DIR = SCENE_DIR  # noqa: F841
print(f'\n[multiview] SCENE_DIR for STEP 5 = {SCENE_DIR}')


In [ ]:
# @title STEP 5 — Run gs-extract-mesh (~5-10 min on T4)
# @markdown The main extraction. Renders depth maps from the 3DGS cameras and
# @markdown fuses them with TSDF into a single triangle mesh. Single CLI call.

import os, sys, time, subprocess
os.chdir('/content/gaustudio')

SCENE_DIR = globals().get('OUTPUT_DIR', '/content/gau_out/triposplat_bridge')
EXTRACT_OUT = f'{SCENE_DIR}/extracted_mesh'
os.makedirs(EXTRACT_OUT, exist_ok=True)

RESOLUTION = 2  # @param {type:"slider", min:1, max:4, step:1}
# @markdown *Render resolution. 1 = full, 2 = half (faster, default), 4 = quarter (fastest, lower quality).*
CLEAN = True  # @param {type:"boolean"}
# @markdown *Run Open3D connected-component cleanup on the extracted mesh (removes small floating blobs).*
VOXEL_SIZE = 0.01  # @param {type:"slider", min:0.001, max:0.05, step:0.001}
# @markdown *Voxel size for TSDF integration in world units. Smaller = more detail, slower.*
SDF_TRUNC = 0.04  # @param {type:"slider", min:0.01, max:0.2, step:0.01}
# @markdown *TSDF truncation distance in world units. Larger = smoother.*

t0 = time.time()
print(f'[extract] Running gs-extract-mesh on {SCENE_DIR}', flush=True)
print(f'[extract] Output: {EXTRACT_OUT}', flush=True)
print(f'[extract] resolution={RESOLUTION} voxel={VOXEL_SIZE} sdf_trunc={SDF_TRUNC} clean={CLEAN}', flush=True)
print(flush=True)

cmd = [
    'gs-extract-mesh',
    '--source-path', SCENE_DIR,
    '--output-path', EXTRACT_OUT,
    '--resolution', str(RESOLUTION),
    '--voxel-size', str(VOXEL_SIZE),
    '--sdf-trunc', str(SDF_TRUNC),
]
if CLEAN:
    cmd.append('--clean')
print(f'[extract] CMD: {" ".join(cmd)}', flush=True)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             universal_newlines=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
if process.returncode != 0:
    print(f'\n[extract] gs-extract-mesh FAILED (returncode={process.returncode})')
    print('[extract] Common issues:')
    print('[extract]   - No cameras in cameras.json (re-run STEP 3 with a valid TripoSplat image)')
    print('[extract]   - Rasterizer import error (re-run STEP 1)')
    print('[extract]   - vdbfusion import error (re-run STEP 1)')
    raise SystemExit(1)
print(f'\n[extract] Done in {(time.time()-t0)/60:.1f} min', flush=True)
print(f'[extract] Output mesh should be at: {EXTRACT_OUT}/fused_mesh.ply', flush=True)
print(f'[extract] Move on to STEP 6 (verify + game-asset transform).')


In [ ]:
# @title STEP 6 — Verify + Game-Asset Transform + Export to .glb
# @markdown Loads the extracted mesh, applies a game-asset transform (scale to
# @markdown meters, base-pivot center), and exports .obj / .glb / .ply for
# @markdown game engines.

import os, sys, time, pathlib
import numpy as np
import trimesh
import open3d as o3d

SCENE_DIR = globals().get('OUTPUT_DIR', '/content/gau_out/triposplat_bridge')
EXTRACT_OUT = f'{SCENE_DIR}/extracted_mesh'
GAME_OUT_DIR = pathlib.Path(SCENE_DIR) / 'game_ready'
GAME_OUT_DIR.mkdir(parents=True, exist_ok=True)
SCENE_NAME = pathlib.Path(SCENE_DIR).name

# Locate the extracted mesh
mesh_candidates = [
    pathlib.Path(EXTRACT_OUT) / 'fused_mesh.ply',
    pathlib.Path(EXTRACT_OUT) / 'mesh.ply',
]
MESH_PLY = next((p for p in mesh_candidates if p.exists()), None)
if MESH_PLY is None:
    # Search recursively
    pl = list(pathlib.Path(EXTRACT_OUT).rglob('*.ply'))
    if pl:
        MESH_PLY = pl[0]
    else:
        raise SystemExit(f'[verify] No .ply found in {EXTRACT_OUT}. STEP 5 may have failed.')
print(f'[verify] Loading {MESH_PLY}')
mesh = trimesh.load(str(MESH_PLY), force='mesh')
if not isinstance(mesh, trimesh.Trimesh):
    raise SystemExit(f'[verify] {MESH_PLY} is not a triangular mesh')
n_v0, n_f0 = len(mesh.vertices), len(mesh.faces)
print(f'[verify] Loaded: {n_v0:,} verts / {n_f0:,} faces')

# ── Game-asset transform ───────────────────────────────────────────
TARGET_SIZE = 1.0  # @param {type:"slider", min:0.05, max:5.0, step:0.05}
# @markdown *Longest-edge target in meters. 0.3=toy, 1.0=human, 1.8=vehicle.*
CENTER_MODE = 'base'  # @param ["base", "origin", "keep"]
# @markdown *base = bbox bottom at origin (actor pivot). origin = center at origin.*
CLEAN_DEGENERATE = True  # @param {type:"boolean"}
# @markdown *Remove degenerate faces (zero-area, duplicate). Recommended.*
WELD_DUPLICATE_VERTS = True  # @param {type:"boolean"}
# @markdown *Merge duplicate vertices within 1e-5 distance. Recommended for game engines.*

if CLEAN_DEGENERATE:
    mesh.remove_degenerate_faces()
    mesh.remove_unreferenced_vertices()
    n_f1 = len(mesh.faces)
    print(f'[clean] Removed degenerate faces: {n_f0:,} -> {n_f1:,}')
if WELD_DUPLICATE_VERTS:
    n_v_pre = len(mesh.vertices)
    mesh.merge_vertices()
    n_v_post = len(mesh.vertices)
    print(f'[clean] Merged duplicate verts: {n_v_pre:,} -> {n_v_post:,}')

if TARGET_SIZE > 0 and CENTER_MODE != 'keep':
    bbox_min = mesh.vertices.min(axis=0)
    bbox_max = mesh.vertices.max(axis=0)
    bbox_center = (bbox_min + bbox_max) / 2.0
    bbox_extent = float((bbox_max - bbox_min).max())
    if bbox_extent > 1e-9:
        scale = TARGET_SIZE / bbox_extent
    else:
        scale = 1.0
    if CENTER_MODE == 'origin':
        translate = -bbox_center
    elif CENTER_MODE == 'base':
        translate = -np.array([bbox_center[0], bbox_min[1], bbox_center[2]])
    mesh.vertices = (mesh.vertices + translate) * scale
    new_extent = float((mesh.vertices.max(axis=0) - mesh.vertices.min(axis=0)).max())
    print(f'[transform] scale={scale:.3f} translate={translate.round(3).tolist()}')
    print(f'[transform] New bbox extent: {new_extent:.3f}m')

# ── Export ────────────────────────────────────────────────────
game_obj = GAME_OUT_DIR / f'{SCENE_NAME}_game.obj'
game_glb = GAME_OUT_DIR / f'{SCENE_NAME}_game.glb'
game_ply = GAME_OUT_DIR / f'{SCENE_NAME}_game.ply'

mesh.export(str(game_obj), file_type='obj')
print(f'[export] Saved: {game_obj}')
try:
    mesh.export(str(game_glb), file_type='glb')
    print(f'[export] Saved: {game_glb}')
except Exception as e:
    print(f'[export] GLB failed: {e} (obj should still work)')
mesh.export(str(game_ply), file_type='ply')
print(f'[export] Saved: {game_ply}')

n_vf = len(mesh.vertices), len(mesh.faces)
print(f'\n[Done] Game-ready mesh: {n_vf[0]:,} verts / {n_vf[1]:,} faces')
print(f'[Done] Files in {GAME_OUT_DIR}/')
print(f'[Done] Move on to STEP 7 (Drive mirror).')


In [ ]:
# @title STEP 7 — Keep-Alive (anti-disconnect)
# @markdown Standard pattern. GauStudio extraction is ~5-10 min, but if you
# @markdown have multiple scenes queued or multi-view input, this prevents
# @markdown Colab from disconnecting you mid-run.

import IPython
from google.colab import output
KEEP_ALIVE_JS = """
function KeepAlive() {
    setInterval(() => {
        console.log("keep-alive:", new Date().toISOString());
        google.colab.kernel.proxyProxy(0).then(() => {}).catch(() => {});
    }, 30 * 60 * 1000);
}
KeepAlive();
"""
display(IPython.display.Javascript(KEEP_ALIVE_JS))
print('[KeepAlive] Started — pings every 30 min.')


In [ ]:
# @title STEP 8 — Copy to Drive
# @markdown Mirrors the game-ready mesh + raw GauStudio outputs to your Drive at
# @markdown `/content/drive/MyDrive/AEI_3D_Out/GauStudio/<scene_name>/`.

import os, shutil, pathlib
from IPython.display import FileLink, display

SCENE_DIR = globals().get('OUTPUT_DIR', '/content/gau_out/triposplat_bridge')
SCENE_NAME = pathlib.Path(SCENE_DIR).name
GAME_OUT_DIR = pathlib.Path(SCENE_DIR) / 'game_ready'
EXTRACT_OUT = pathlib.Path(SCENE_DIR) / 'extracted_mesh'
DRIVE_BASE = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/GauStudio')

if not DRIVE_BASE.parent.exists():
    print('[drive] /content/drive not mounted. Skipping Drive mirror.')
    print('[drive]   Mount Drive for persistence: from the Colab file browser, click the Drive icon.')
else:
    # Game-ready output (the one you actually want to use)
    game_dst = DRIVE_BASE / 'game_ready' / SCENE_NAME
    if GAME_OUT_DIR.exists():
        game_dst.mkdir(parents=True, exist_ok=True)
        for f in GAME_OUT_DIR.iterdir():
            if f.is_file():
                shutil.copy2(f, game_dst / f.name)
        n = sum(1 for _ in game_dst.iterdir())
        print(f'[drive] Mirrored {n} game-ready files to {game_dst}')
    # Raw GauStudio outputs (intermediate fused_mesh.ply, cameras.json, etc.)
    raw_dst = DRIVE_BASE / 'raw_gau' / SCENE_NAME
    if EXTRACT_OUT.exists():
        raw_dst.mkdir(parents=True, exist_ok=True)
        for f in EXTRACT_OUT.rglob('*'):
            if f.is_file() and f.stat().st_size < 200_000_000:
                rel = f.relative_to(EXTRACT_OUT)
                dst = raw_dst / rel
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dst)
        n = sum(1 for _ in raw_dst.rglob('*') if _.is_file())
        print(f'[drive] Mirrored {n} raw GauStudio files to {raw_dst}')
    # Show download links
    print('\n[drive] Download links:')
    if GAME_OUT_DIR.exists():
        for f in sorted(GAME_OUT_DIR.iterdir()):
            if f.is_file():
                display(FileLink(str(f)))
                print(f'  {f.name} ({f.stat().st_size/1024/1024:.1f} MB)')


In [ ]:
# @title STEP 9 — Help / License / Known Issues / Citation
# @markdown Reference material. Read this if anything in the pipeline went wrong.

help_md = '''
# GauStudio — Help, License, and Known Issues

## License (READ THIS)

**GauStudio uses a MIXED license:**

- **Main framework (Python code, configs, scripts):** [MIT](https://github.com/GAP-LAB-CUHK-SZ/gaustudio/blob/main/LICENSE)
- **`submodules/gaustudio-diff-gaussian-rasterization`:** derivative of [INRIA's 3DGS rasterizer](https://github.com/graphdeco-inria/gaussian-splatting) — inherits INRIA's **custom non-commercial research license**

**Practical effect:** free for academic and personal-asset research/evaluation. **NOT for commercial use** without:
1. Explicit permission from INRIA (`stip-sophia.transfert@inria.fr`), AND
2. Permission from the GauStudio authors (CJ Ye et al., GAP Lab @ CUHK-Shenzhen)

For commercial assets, use [Kiri Engine 3DGS-to-Mesh](https://www.kiriengine.app/blog/what-is-3dgs-to-mesh), [Polycam](https://poly.cam/), or [LumaGen](https://lumalabs.ai/gen).

## Pipeline overview

```
TripoSplat (image → 3DGS PLY)  →  Step 3 bridge (PLY → cameras.json)  →  Step 5 (gs-extract-mesh CLI)  →  Step 6 (game-asset transform + .glb)  →  Step 8 (Drive)
```

## Compute

| GPU | VRAM | Time (single-image bridge) | Time (multi-view) |
|-----|------|---------------------------|-------------------|
| T4  | 15 GB | ~5-10 min | ~10-20 min |
| L4  | 22 GB | ~3-5 min | ~5-10 min |
| A100 | 40 GB | ~2-3 min | ~3-5 min |

## Known issues

1. **`diff_gaussian_rasterization` import fails**: rasterizer .so doesn't match torch ABI. Fix: re-run STEP 1 (it rebuilds against active torch).
2. **`vdbfusion` import fails**: source build failed. Fix: re-run STEP 1; if PyPI fallback also fails, install Open3D as a fallback (still does TSDF, just slower).
3. **`gs-extract-mesh: command not found`**: gauStudio develop install didn't run. Fix: re-run STEP 1.
4. **No cameras in scene** (extraction yields empty mesh): cameras.json is empty or your scene has 0 images. Fix: re-run STEP 3 with a valid TripoSplat image.
5. **Mesh has holes in subject interior**: too few input views. For 1-image input, holes are expected. For multi-view with 3+ overlapping cameras, this shouldn't happen. Use STEP 4 instead.
6. **Single-image mesh quality is modest**: same caveat as SuGaR. GauStudio's TSDF fusion + rendered depths is fundamentally limited without multi-view consistency. For best quality, use multi-view input (STEP 4).

## When to use GauStudio vs SuGaR

| | GauStudio (this notebook) | SuGaR |
|--|--------------------------|-------|
| Time per scene | ~5-10 min | ~2-3 hrs |
| VRAM | T4 15 GB OK | L4 22 GB required |
| Mesh quality (single-view) | Modest | Modest (similar) |
| Mesh quality (multi-view) | Good | Best |
| Algorithm | TSDF fusion of depth maps | Density-field + Poisson |
| Topological style | Smoother, cleaner | Sharper, more detail |
| Best for | Quick sweeps, low-LOD game assets | Hero assets, CAD-like outputs |

## Citation

```bibtex
@InProceedings{Ye2024GauStudio,
    author    = {Ye, Chongjie and Danelljan, Martin and Yu, Fisher and Ke, Lanqing},
    title     = {{GauStudio: A Modular Framework for 3D Gaussian Splatting},
    booktitle = {CVPR},
    year      = {2024},
}
```

## See also

- **TripoSplat_Colab.ipynb** — generate the 3DGS PLY from a single image
- **SuGaR_Colab.ipynb** — alternative high-quality mesh extraction (slower, sharper)
- **Mesh_Optimizer_Colab.ipynb** — post-process either output (UV unwrap, fill holes, etc.)
- **Step 8 of TripoSplat_Colab.ipynb** — standalone post-processor for any folder of `*_mesh.ply` files
'''
from IPython.display import Markdown, display
display(Markdown(help_md))
print(help_md)
